# 02b — Team Standings & Starting Pitcher Features
Adds features that a ticket buyer sees on the MLB app when deciding whether to go to a game: team record, win/loss streak, division standing, and starting pitcher ERA (both teams) as of the day of the game.

Only computed for games with `status == 'Final'` — for scheduled future games we don't know the actual standings or confirmed starters yet.

In [1]:
import pandas as pd
import numpy as np
import requests
import time

TEAM_ID = 121  # New York Mets
ACE_ERA_THRESHOLD = 3.00  # placeholder "stellar" cutoff — adjust once you've settled on a number

df = pd.read_csv("../data/mets_master.csv")
df["date"] = pd.to_datetime(df["date"]).dt.strftime("%Y-%m-%d")
print(f"Loaded {len(df)} games from mets_master.csv")

df_2026 = pd.read_csv("../data/mets_2026_prediction.csv")
df_2026_final = df_2026[df_2026["status"] == "Final"].copy()
print(f"Loaded {len(df_2026_final)} completed 2026 games")

# Team name -> id lookup, needed to look up opponent standings by name
teams_resp = requests.get("https://statsapi.mlb.com/api/v1/teams", params={"sportId": 1}, timeout=15).json()
TEAM_NAME_TO_ID = {t["name"]: t["id"] for t in teams_resp["teams"]}
print(f"Loaded {len(TEAM_NAME_TO_ID)} team ids")

Loaded 320 games from mets_master.csv
Loaded 49 completed 2026 games
Loaded 30 team ids


## Standings: Record, Streak, Games Back
One API call per unique date returns every team's record as of that date — cache by date since both the Mets and the opponent's row come from the same response.

In [2]:
STANDINGS_CACHE = {}

def get_standings_for_date(date):
    if date in STANDINGS_CACHE:
        return STANDINGS_CACHE[date]
    url = "https://statsapi.mlb.com/api/v1/standings"
    params = {"leagueId": "103,104", "season": date[:4], "date": date}
    data = {}
    for attempt in range(3):
        try:
            data = requests.get(url, params=params, timeout=10).json()
            break
        except requests.exceptions.RequestException:
            if attempt == 2:
                data = {}
            else:
                time.sleep(2)

    teams = {}
    for record in data.get("records", []):
        for tr in record.get("teamRecords", []):
            team_id = tr["team"]["id"]
            streak = tr.get("streak") or {}
            streak_num = streak.get("streakNumber")
            if streak_num is not None and streak.get("streakType") == "losses":
                streak_num = -streak_num
            gb_raw = tr.get("gamesBack")
            games_back = 0.0 if gb_raw in ("-", None) else float(gb_raw)
            lr = tr.get("leagueRecord", {})
            teams[team_id] = {
                "wins": lr.get("wins"),
                "losses": lr.get("losses"),
                "win_pct": float(lr.get("pct") or 0),
                "streak_code": streak.get("streakCode"),
                "streak_num": streak_num,
                "games_back": games_back,
            }
    STANDINGS_CACHE[date] = teams
    return teams

def add_standings_columns(frame):
    frame = frame.copy()
    dates = frame["date"].unique()
    print(f"Pulling standings for {len(dates)} unique dates...")
    for i, d in enumerate(dates):
        get_standings_for_date(d)
        if (i + 1) % 25 == 0 or (i + 1) == len(dates):
            print(f"  {i + 1}/{len(dates)} dates pulled")

    mets_cols = {"wins": [], "losses": [], "win_pct": [], "streak_code": [], "streak_num": [], "games_back": []}
    opp_cols = {"wins": [], "losses": [], "win_pct": [], "streak_code": [], "streak_num": [], "games_back": []}
    for _, row in frame.iterrows():
        standings = STANDINGS_CACHE.get(row["date"], {})
        mets = standings.get(TEAM_ID, {})
        opp_id = TEAM_NAME_TO_ID.get(row["opponent"])
        opp = standings.get(opp_id, {}) if opp_id else {}
        for key in mets_cols:
            mets_cols[key].append(mets.get(key))
            opp_cols[key].append(opp.get(key))

    for key in mets_cols:
        frame[f"mets_{key}"] = mets_cols[key]
        frame[f"opp_{key}"] = opp_cols[key]
    return frame

df = add_standings_columns(df)
print(df[["date", "opponent", "mets_wins", "mets_losses", "mets_streak_code", "mets_games_back",
          "opp_wins", "opp_losses", "opp_streak_code", "opp_games_back"]].head())

Pulling standings for 307 unique dates...
  25/307 dates pulled
  50/307 dates pulled
  75/307 dates pulled
  100/307 dates pulled
  125/307 dates pulled
  150/307 dates pulled
  175/307 dates pulled
  200/307 dates pulled
  225/307 dates pulled
  250/307 dates pulled
  275/307 dates pulled
  300/307 dates pulled
  307/307 dates pulled
         date              opponent  mets_wins  mets_losses mets_streak_code  \
0  2022-04-15  Arizona Diamondbacks          6            2               W3   
1  2022-04-16  Arizona Diamondbacks          6            3               L1   
2  2022-04-17  Arizona Diamondbacks          7            3               W1   
3  2022-04-19  San Francisco Giants          9            3               W3   
4  2022-04-19  San Francisco Giants          9            3               W3   

   mets_games_back  opp_wins  opp_losses opp_streak_code  opp_games_back  
0              0.0       2.0         5.0              L1             3.0  
1              0.0       3.0   

In [3]:
df_2026_final = add_standings_columns(df_2026_final)
print(df_2026_final[["date", "opponent", "mets_wins", "mets_losses", "mets_streak_code", "mets_games_back"]].head())

Pulling standings for 47 unique dates...
  25/47 dates pulled
  47/47 dates pulled
         date              opponent  mets_wins  mets_losses mets_streak_code  \
0  2026-03-26    Pittsburgh Pirates          1            0               W1   
1  2026-03-28    Pittsburgh Pirates          2            0               W2   
2  2026-03-29    Pittsburgh Pirates          2            1               L1   
3  2026-04-07  Arizona Diamondbacks          7            4               W4   
4  2026-04-08  Arizona Diamondbacks          7            5               L1   

   mets_games_back  
0              0.0  
1              0.0  
2              1.0  
3              0.0  
4              0.5  


## Starting Pitchers & ERA Entering the Game
The Stats API only reports season-cumulative stats as of *today*, not a snapshot as of a past date. To get each starter's ERA *entering* a given game, pull their full game log for the season and sum earned runs / innings pitched only from starts strictly before that date.

In [4]:
def get_probable_pitchers(team_id, season):
    url = "https://statsapi.mlb.com/api/v1/schedule"
    params = {"sportId": 1, "season": season, "gameType": "R", "teamId": team_id, "hydrate": "probablePitcher"}
    data = requests.get(url, params=params, timeout=15).json()
    rows = []
    for date in data.get("dates", []):
        for game in date.get("games", []):
            if game["teams"]["home"]["team"]["id"] != team_id:
                continue
            home_p = game["teams"]["home"].get("probablePitcher") or {}
            away_p = game["teams"]["away"].get("probablePitcher") or {}
            rows.append({
                "game_pk": game["gamePk"],
                "date": date["date"],
                "season": season,
                "mets_starter_id": home_p.get("id"),
                "mets_starter_name": home_p.get("fullName"),
                "opp_starter_id": away_p.get("id"),
                "opp_starter_name": away_p.get("fullName"),
            })
    return rows

pitcher_rows = []
for season in [2022, 2023, 2024, 2025, 2026]:
    rows = get_probable_pitchers(TEAM_ID, season)
    pitcher_rows.extend(rows)
    print(f"{season}: {len(rows)} games with probable pitcher data")

df_pitchers = pd.DataFrame(pitcher_rows)

# Postponed/suspended games can be listed twice (original date + makeup date) with
# different probable-pitcher snapshots for the same game_pk — keep the row with the
# latest date, which reflects the starter closer to when the game was actually played
n_before = len(df_pitchers)
df_pitchers = df_pitchers.sort_values("date").drop_duplicates(subset="game_pk", keep="last")
print(f"Dropped {n_before - len(df_pitchers)} duplicate game_pk listings (postponed/makeup games)")

print(f"\nTotal: {len(df_pitchers)} rows")
print(f"Missing Mets starter: {df_pitchers['mets_starter_id'].isna().sum()}")
print(f"Missing opponent starter: {df_pitchers['opp_starter_id'].isna().sum()}")
df_pitchers.head()

2022: 84 games with probable pitcher data
2023: 88 games with probable pitcher data
2024: 85 games with probable pitcher data
2025: 82 games with probable pitcher data
2026: 83 games with probable pitcher data
Dropped 17 duplicate game_pk listings (postponed/makeup games)

Total: 405 rows
Missing Mets starter: 29
Missing opponent starter: 29


,game_pk,date,season,mets_starter_id,mets_starter_name,opp_starter_id,opp_starter_name
0,662485,2022-04-15,2022,605135.0,Chris Bassitt,605200.0,Zach Davies
1,662626,2022-04-16,2022,471911.0,Carlos Carrasco,668678.0,Zac Gallen
2,662625,2022-04-17,2022,656849.0,David Peterson,528748.0,Humberto Castellanos
4,662350,2022-04-19,2022,656731.0,Tylor Megill,502171.0,Alex Cobb
5,662616,2022-04-19,2022,453286.0,Max Scherzer,657277.0,Logan Webb


In [5]:
def ip_to_innings_decimal(ip_str):
    if ip_str in (None, ""):
        return 0.0
    whole, _, frac = str(ip_str).partition(".")
    thirds = {"": 0, "0": 0, "1": 1, "2": 2}.get(frac, 0)
    return int(whole) + thirds / 3

GAMELOG_CACHE = {}

def get_gamelog(pitcher_id, season):
    key = (pitcher_id, season)
    if key in GAMELOG_CACHE:
        return GAMELOG_CACHE[key]
    url = f"https://statsapi.mlb.com/api/v1/people/{pitcher_id}/stats"
    params = {"stats": "gameLog", "group": "pitching", "season": season}
    starts = []
    for attempt in range(3):
        try:
            data = requests.get(url, params=params, timeout=10).json()
            for split in data.get("stats", [{}])[0].get("splits", []):
                if split["stat"].get("gamesStarted", 0) != 1:
                    continue
                starts.append({
                    "date": split["date"],
                    "earned_runs": split["stat"].get("earnedRuns", 0),
                    "ip": ip_to_innings_decimal(split["stat"].get("inningsPitched")),
                })
            break
        except (requests.exceptions.RequestException, IndexError, KeyError):
            if attempt == 2:
                starts = []
            else:
                time.sleep(2)
    starts.sort(key=lambda s: s["date"])
    GAMELOG_CACHE[key] = starts
    return starts

def era_entering_game(pitcher_id, season, game_date):
    if pd.isna(pitcher_id):
        return None
    starts = get_gamelog(int(pitcher_id), int(season))
    prior = [s for s in starts if s["date"] < game_date]
    total_er = sum(s["earned_runs"] for s in prior)
    total_ip = sum(s["ip"] for s in prior)
    if total_ip == 0:
        return None
    return round(9 * total_er / total_ip, 2)

unique_pitcher_seasons = pd.concat([
    df_pitchers[["mets_starter_id", "season"]].rename(columns={"mets_starter_id": "id"}),
    df_pitchers[["opp_starter_id", "season"]].rename(columns={"opp_starter_id": "id"}),
]).dropna().drop_duplicates()
print(f"Fetching game logs for {len(unique_pitcher_seasons)} unique pitcher-seasons...")

for i, (_, row) in enumerate(unique_pitcher_seasons.iterrows()):
    get_gamelog(int(row["id"]), int(row["season"]))
    if (i + 1) % 25 == 0 or (i + 1) == len(unique_pitcher_seasons):
        print(f"  {i + 1}/{len(unique_pitcher_seasons)} pitcher-seasons pulled")

df_pitchers["mets_starter_era"] = df_pitchers.apply(
    lambda r: era_entering_game(r["mets_starter_id"], r["season"], r["date"]), axis=1)
df_pitchers["opp_starter_era"] = df_pitchers.apply(
    lambda r: era_entering_game(r["opp_starter_id"], r["season"], r["date"]), axis=1)
df_pitchers["starter_era_diff"] = df_pitchers["opp_starter_era"] - df_pitchers["mets_starter_era"]
df_pitchers["mets_starter_is_ace"] = (df_pitchers["mets_starter_era"] < ACE_ERA_THRESHOLD).astype("Int64")

print(f"\nGames with a computed Mets starter ERA: {df_pitchers['mets_starter_era'].notna().sum()} / {len(df_pitchers)}")
print("(NaN means it was that pitcher's first start of the season — no prior starts to compute ERA from)")
df_pitchers[["date", "mets_starter_name", "mets_starter_era", "opp_starter_name", "opp_starter_era", "starter_era_diff"]].head(10)

Fetching game logs for 401 unique pitcher-seasons...
  25/401 pitcher-seasons pulled
  50/401 pitcher-seasons pulled
  75/401 pitcher-seasons pulled
  100/401 pitcher-seasons pulled
  125/401 pitcher-seasons pulled
  150/401 pitcher-seasons pulled
  175/401 pitcher-seasons pulled
  200/401 pitcher-seasons pulled
  225/401 pitcher-seasons pulled
  250/401 pitcher-seasons pulled
  275/401 pitcher-seasons pulled
  300/401 pitcher-seasons pulled
  325/401 pitcher-seasons pulled
  350/401 pitcher-seasons pulled
  375/401 pitcher-seasons pulled
  400/401 pitcher-seasons pulled
  401/401 pitcher-seasons pulled

Games with a computed Mets starter ERA: 353 / 405
(NaN means it was that pitcher's first start of the season — no prior starts to compute ERA from)


,date,mets_starter_name,mets_starter_era,opp_starter_name,opp_starter_era,starter_era_diff
0,2022-04-15,Chris Bassitt,0.00,Zach Davies,3.60,3.60
1,2022-04-16,Carlos Carrasco,1.59,Zac Gallen,NaN,NaN
2,2022-04-17,David Peterson,NaN,Humberto Castellanos,NaN,NaN
4,2022-04-19,Tylor Megill,0.00,Alex Cobb,3.60,3.60
5,2022-04-19,Max Scherzer,3.27,Logan Webb,1.29,-1.98
6,2022-04-20,Chris Bassitt,0.75,Carlos Rodón,1.50,0.75
7,2022-04-21,Carlos Carrasco,0.84,Anthony DeSclafani,4.32,3.48
8,2022-04-29,Tylor Megill,2.35,Aaron Nola,3.74,1.39
9,2022-04-30,Taijuan Walker,0.00,Kyle Gibson,3.47,3.47
10,2022-05-01,Max Scherzer,1.80,Zach Eflin,3.20,1.40


## Merge Into Master Dataset

In [6]:
pitcher_cols = ["game_pk", "mets_starter_name", "mets_starter_era", "opp_starter_name",
                "opp_starter_era", "starter_era_diff", "mets_starter_is_ace"]

df = df.merge(df_pitchers[pitcher_cols], on="game_pk", how="left")
df.to_csv("../data/mets_master.csv", index=False)
print(f"Saved mets_master.csv with {df.shape[1]} columns, {len(df)} rows")
print(list(df.columns))

Saved mets_master.csv with 32 columns, 320 rows
['date', 'season', 'opponent', 'home_score', 'away_score', 'attendance', 'status', 'game_pk', 'avg_temp_f', 'avg_precip_mm', 'n_promotions', 'is_promo', 'is_bobblehead', 'promotion_names', 'mets_wins', 'opp_wins', 'mets_losses', 'opp_losses', 'mets_win_pct', 'opp_win_pct', 'mets_streak_code', 'opp_streak_code', 'mets_streak_num', 'opp_streak_num', 'mets_games_back', 'opp_games_back', 'mets_starter_name', 'mets_starter_era', 'opp_starter_name', 'opp_starter_era', 'starter_era_diff', 'mets_starter_is_ace']


In [7]:
df_2026_final = df_2026_final.merge(df_pitchers[pitcher_cols], on="game_pk", how="left")
df_2026_final.to_csv("../data/mets_2026_standings_pitchers.csv", index=False)
print(f"Saved mets_2026_standings_pitchers.csv with {df_2026_final.shape[1]} columns, {len(df_2026_final)} rows")

Saved mets_2026_standings_pitchers.csv with 26 columns, 49 rows
